# IMC Prosperity 4 — Round 5: 50-Product Quantitative Analysis

**Goal:** Discover cointegration, mean-reverting spreads, and lead-lag relationships across 50 products in 10 categories. Build tradeable z-score signals under a strict position limit of ±10.

**Two-step pair trading framework:**
1. **Pair Selection** — statistical tests (ADF, Engle-Granger, half-life, Hurst exponent)
2. **Pair Trading** — z-score entry/exit with position limit enforcement

In [ ]:
# %%
# ─── CELL 1: Imports & Configuration ────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from statsmodels.tsa.stattools import adfuller, coint, ccf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})
sns.set_style('darkgrid')

DATA_DIR = '/Users/davidmarco/Desktop/IMC Prosperity/ROUND 5/ROUND_5_DATA'
DAYS = [2, 3, 4]
POS_LIMIT = 10
ROUND_NUM = 5

CATEGORIES = {
    'Galaxy Sounds':  ['GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_BLACK_HOLES',
                       'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_WINDS',
                       'GALAXY_SOUNDS_SOLAR_FLAMES'],
    'Sleep Pods':     ['SLEEP_POD_SUEDE', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_POLYESTER',
                       'SLEEP_POD_NYLON', 'SLEEP_POD_COTTON'],
    'Microchips':     ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE',
                       'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE'],
    'Pebbles':        ['PEBBLES_XS', 'PEBBLES_S', 'PEBBLES_M', 'PEBBLES_L', 'PEBBLES_XL'],
    'Robots':         ['ROBOT_VACUUMING', 'ROBOT_MOPPING', 'ROBOT_DISHES',
                       'ROBOT_LAUNDRY', 'ROBOT_IRONING'],
    'UV Visors':      ['UV_VISOR_YELLOW', 'UV_VISOR_AMBER', 'UV_VISOR_ORANGE',
                       'UV_VISOR_RED', 'UV_VISOR_MAGENTA'],
    'Translators':    ['TRANSLATOR_SPACE_GRAY', 'TRANSLATOR_ASTRO_BLACK',
                       'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST',
                       'TRANSLATOR_VOID_BLUE'],
    'Panels':         ['PANEL_1X2', 'PANEL_2X2', 'PANEL_1X4', 'PANEL_2X4', 'PANEL_4X4'],
    'Shakes':         ['OXYGEN_SHAKE_MORNING_BREATH', 'OXYGEN_SHAKE_EVENING_BREATH',
                       'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_CHOCOLATE', 'OXYGEN_SHAKE_GARLIC'],
    'Snacks':         ['SNACKPACK_CHOCOLATE', 'SNACKPACK_VANILLA', 'SNACKPACK_PISTACHIO',
                       'SNACKPACK_STRAWBERRY', 'SNACKPACK_RASPBERRY'],
}

# ── Panels have a natural area ordering worth knowing ──
PANEL_AREAS = {'PANEL_1X2': 2, 'PANEL_2X2': 4, 'PANEL_1X4': 4, 'PANEL_2X4': 8, 'PANEL_4X4': 16}
# PANEL_2X2 and PANEL_1X4 share the same area (4) — prime cointegration candidate

print('Categories defined. Total products:', sum(len(v) for v in CATEGORIES.values()))

Categories defined. Total products: 50


In [2]:
# %%
# ─── CELL 2: Data Ingestion & Cleaning ──────────────────────────────────────

def load_prices(data_dir: str, days: list) -> pd.DataFrame:
    """
    Load all price CSVs, concatenate, and pivot to wide format.
    Returns: DataFrame indexed by (day * 1_000_000 + timestamp) with one
             column per product containing mid_price.
    """
    frames = []
    for day in days:
        path = os.path.join(data_dir, f'prices_round_{ROUND_NUM}_day_{day}.csv')
        df = pd.read_csv(path, sep=';')
        # Create a global time index: day offset ensures days don't overlap
        df['global_ts'] = day * 1_000_000 + df['timestamp']
        frames.append(df[['global_ts', 'day', 'timestamp', 'product', 'mid_price',
                           'bid_price_1', 'ask_price_1']])

    raw = pd.concat(frames, ignore_index=True)
    raw = raw.sort_values(['product', 'global_ts'])

    # Pivot: rows = global_ts, columns = product, values = mid_price
    pivot = raw.pivot_table(index='global_ts', columns='product', values='mid_price')
    pivot.columns.name = None
    pivot = pivot.sort_index()

    # Also store bid/ask for spread analysis
    bid_pivot = raw.pivot_table(index='global_ts', columns='product', values='bid_price_1')
    bid_pivot.columns.name = None
    ask_pivot = raw.pivot_table(index='global_ts', columns='product', values='ask_price_1')
    ask_pivot.columns.name = None

    print(f'Loaded {len(pivot):,} timestamps × {pivot.shape[1]} products')
    print(f'Days covered: {sorted(raw["day"].unique())}')
    print(f'Missing values: {pivot.isna().sum().sum()}')
    return pivot, bid_pivot, ask_pivot, raw


prices, bids, asks, raw_df = load_prices(DATA_DIR, DAYS)

# Separate per-day slices for stationarity checks on individual days
day_slices = {}
for day in DAYS:
    mask = (prices.index >= day * 1_000_000) & (prices.index < (day + 1) * 1_000_000)
    day_slices[day] = prices.loc[mask]

# Category views: dict of {cat_name -> DataFrame with only those 5 columns}
cat_prices = {cat: prices[cols].copy() for cat, cols in CATEGORIES.items()}

prices.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/davidmarco/Desktop/IMC Prosperity/ROUND 5/ROUND_5_DATA/prices_round_5_day_2.csv'

In [ ]:
# %%
# ─── CELL 3: Summary Statistics ─────────────────────────────────────────────

def summary_stats(prices_wide: pd.DataFrame, categories: dict) -> pd.DataFrame:
    stats_rows = []
    for cat, cols in categories.items():
        for col in cols:
            s = prices_wide[col].dropna()
            stats_rows.append({
                'category':   cat,
                'product':    col,
                'mean':       s.mean(),
                'std':        s.std(),
                'min':        s.min(),
                'max':        s.max(),
                'range':      s.max() - s.min(),
                'cv_%':       100 * s.std() / s.mean(),   # coefficient of variation
                'ret_autocorr': s.diff().autocorr(1),     # return autocorrelation
            })
    df = pd.DataFrame(stats_rows).set_index(['category', 'product'])
    return df.round(4)

stats_df = summary_stats(prices, CATEGORIES)
print('=== Summary Statistics ===')
display(stats_df)
print('\n--- Category-level return autocorr (negative = mean-reverting) ---')
display(stats_df.groupby('category')['ret_autocorr'].mean().sort_values())

In [ ]:
# %%
# ─── CELL 4: EDA — Price Series Plots (All 10 Categories) ───────────────────

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(5, 2, figsize=(20, 30))
axes = axes.flatten()

for idx, (cat, cols) in enumerate(CATEGORIES.items()):
    ax = axes[idx]
    cat_df = prices[cols]

    # Normalise to first value for comparability
    normalised = cat_df.div(cat_df.iloc[0])

    for i, col in enumerate(cols):
        label = col.replace(cat.upper().replace(' ', '_') + '_', '').replace('_', ' ')
        ax.plot(normalised[col].values, color=COLORS[i], lw=0.7, label=label)

    # Mark day boundaries (day transitions at index multiples of 10000)
    for d_bound in [10000, 20000]:
        if d_bound < len(normalised):
            ax.axvline(d_bound, color='black', lw=1.2, ls='--', alpha=0.5)

    ax.set_title(f'{cat}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Normalised price')

plt.suptitle('Normalised Price Series — All 10 Categories (dashed = day boundary)', 
             fontsize=14, fontweight='bold', y=1.002)
plt.tight_layout()
plt.savefig('price_series_all_categories.png', bbox_inches='tight')
plt.show()
print('Saved: price_series_all_categories.png')

In [ ]:
# %%
# ─── CELL 5: Spread Diagnostics Within Each Category ────────────────────────
#
# Before running cointegration tests, plot raw pairwise price differences
# to visually identify mean-reverting spreads.

fig, axes = plt.subplots(5, 2, figsize=(20, 28))
axes = axes.flatten()

for idx, (cat, cols) in enumerate(CATEGORIES.items()):
    ax = axes[idx]
    cat_df = prices[cols]
    
    # Plot all pairwise raw spreads (demeaned) for the first 5 pairs
    pairs = list(itertools.combinations(cols, 2))[:6]
    for i, (a, b) in enumerate(pairs):
        spread = (cat_df[a] - cat_df[b])
        demeaned = spread - spread.mean()
        short_a = a.split('_')[-1]
        short_b = b.split('_')[-1]
        ax.plot(demeaned.values, lw=0.5, alpha=0.8, label=f'{short_a}−{short_b}')

    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_title(f'{cat} — Raw Price Spreads (demeaned)', fontsize=10, fontweight='bold')
    ax.legend(fontsize=6, ncol=2)
    ax.set_xlabel('Time step')

plt.suptitle('Pairwise Price Spreads Within Categories', fontsize=14, y=1.001)
plt.tight_layout()
plt.savefig('spread_diagnostics.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%
# ─── CELL 6: Pure-NumPy Helpers — OLS, TLS, ADF, Half-Life ──────────────────
#
# OLS helpers use closed-form one-regressor formulas. NO statsmodels.OLS,
# no matrix inversion, and no np.linalg least-squares solve for simple OLS.
# adfuller is kept ONLY for the unit-root test.
#
# ALPHA NOTES:
#   • TLS (orthogonal regression via SVD) treats both legs symmetrically.
#     For pair trading where BOTH prices are noisy, TLS often produces a
#     more stationary spread than OLS → directly improves z-score SNR.
#   • Half-life and Hurst together separate "fast mean-reverting alpha"
#     from "slow drift" → only HL ∈ [50, 2000] AND Hurst < 0.45 is alpha.

from numpy.lib.stride_tricks import sliding_window_view


def _simple_ols_alpha_beta(y, x):
    """Closed-form OLS for y = α + β·x."""
    y = np.asarray(y, dtype=float).ravel()
    x = np.asarray(x, dtype=float).ravel()
    x_mean = x.mean()
    y_mean = y.mean()
    x_cent = x - x_mean
    denom = np.dot(x_cent, x_cent)
    if denom <= 1e-12:
        return np.nan, np.nan
    beta = float(np.dot(x_cent, y - y_mean) / denom)
    alpha = float(y_mean - beta * x_mean)
    return alpha, beta


def ols_hedge_ratio(y, x):
    """
    Static OLS via the closed-form one-regressor formula: y = α + β·x.
    Returns: (alpha, beta, spread = y − β·x).
    """
    y_arr = np.asarray(y, dtype=float).ravel()
    x_arr = np.asarray(x, dtype=float).ravel()
    alpha, beta = _simple_ols_alpha_beta(y_arr, x_arr)
    return alpha, beta, y_arr - beta * x_arr


def tls_hedge_ratio(y, x):
    """
    Total Least Squares via SVD (orthogonal / Deming regression).
    Minimizes perpendicular distance from points to the best-fit line —
    treats x and y noise symmetrically, unlike OLS.

    Method: SVD of demeaned (x, y) matrix; the smallest right-singular
    vector is the normal to the regression line.
    """
    y = np.asarray(y, dtype=float).ravel()
    x = np.asarray(x, dtype=float).ravel()
    M = np.column_stack([x - x.mean(), y - y.mean()])    # (N, 2)
    _, _, Vt = np.linalg.svd(M, full_matrices=False)
    nx, ny = Vt[-1]                                       # smallest sing-vector
    beta  = float(-nx / ny) if abs(ny) > 1e-12 else np.inf
    alpha = float(y.mean() - beta * x.mean())
    return alpha, beta, y - beta * x


def adf_test(series):
    """ADF unit-root test (statsmodels.adfuller; kept for canonical p-values)."""
    s = np.asarray(series).ravel()
    s = s[~np.isnan(s)]
    if len(s) < 30:
        return {'adf_stat': np.nan, 'adf_pval': np.nan, 'stationary': False}
    res = adfuller(s, maxlag=20, autolag='AIC', regression='c')
    return {'adf_stat': res[0], 'adf_pval': res[1], 'stationary': res[1] < 0.05}


def half_life(spread):
    """
    Mean-reversion half-life from AR(1):  Δs_t = α + β·s_{t−1}.
    Uses the same closed-form one-regressor OLS formula as hedge ratios.
    Returns ∞ if non-stationary or β ≥ 0.
    """
    s = np.asarray(spread).ravel()
    s = s[~np.isnan(s)]
    if len(s) < 50:
        return np.inf
    delta = np.diff(s)
    lag = s[:-1]
    _, beta = _simple_ols_alpha_beta(delta, lag)
    return float(-np.log(2) / beta) if np.isfinite(beta) and beta < 0 else np.inf


def hurst_exponent(series, max_lag: int = 100) -> float:
    """Vectorized Hurst estimate. H<0.5 mean-reverting, H≈0.5 random walk."""
    s = np.asarray(series, dtype=float).ravel()
    s = s[~np.isnan(s)]
    if len(s) < 200:
        return np.nan

    lags = np.arange(2, min(max_lag, len(s) // 4))
    if len(lags) < 2:
        return np.nan

    # Rectangular lag-difference matrix; invalid tail slots are ignored.
    width = len(s) - lags[0]
    offsets = np.arange(width)
    valid_offsets = offsets[None, :] < (len(s) - lags[:, None])
    lagged_idx = np.minimum(lags[:, None] + offsets[None, :], len(s) - 1)
    diffs = s[lagged_idx] - s[offsets][None, :]
    diffs = np.where(valid_offsets, diffs, np.nan)
    tau = np.nanstd(diffs, axis=1)

    valid = tau > 0
    if valid.sum() < 2:
        return np.nan
    return float(np.polyfit(np.log(lags[valid]), np.log(tau[valid]), 1)[0])


# ── Backwards-compat shim so downstream cells (10, 13, 17-24) keep working ──
def fit_hedge_ratio(y, x):
    """Legacy 2-tuple wrapper around ols_hedge_ratio. Returns (beta, spread as Series)."""
    y_orig = y  # preserve pandas Series index if present
    _, b, sp = ols_hedge_ratio(y, x)
    if isinstance(y_orig, pd.Series):
        sp = pd.Series(sp, index=y_orig.index)
    return b, sp


print('Pure-NumPy helpers defined:')
print('   ols_hedge_ratio, tls_hedge_ratio, adf_test, half_life, hurst_exponent')
print('   fit_hedge_ratio  (legacy shim; returns (beta, spread))')


In [ ]:
# %%
# ─── CELL 7: STEP 1 — Cointegration Search (OLS + TLS, NumPy-only) ──────────
#
# For every within-category pair, fit BOTH OLS and TLS hedge ratios,
# run ADF on each spread, take the better of the two as the canonical
# spread for that pair. Pairs where TLS materially beats OLS are flagged.
#
# ALPHA NOTE: Pairs where TLS dominates OLS in stationarity are PRIMARY
# strategy candidates — they were hidden by the OLS asymmetry assumption.
# Trade them at the TLS hedge ratio.

pair_results = []

for cat, cols in CATEGORIES.items():
    cat_df = prices[cols].dropna()

    for y_col, x_col in itertools.combinations(cols, 2):
        y = cat_df[y_col].values
        x = cat_df[x_col].values

        # ── OLS hedge ratio + diagnostics ──
        _, beta_ols, sp_ols = ols_hedge_ratio(y, x)
        adf_ols = adf_test(sp_ols)
        hl_ols  = half_life(sp_ols)

        # ── TLS hedge ratio + diagnostics ──
        _, beta_tls, sp_tls = tls_hedge_ratio(y, x)
        adf_tls = adf_test(sp_tls)
        hl_tls  = half_life(sp_tls)

        # ── Hurst on the better of the two ──
        better_tls  = adf_tls['adf_pval'] < adf_ols['adf_pval']
        best_spread = sp_tls if better_tls else sp_ols
        hu          = hurst_exponent(best_spread)

        pair_results.append({
            'category':   cat,
            'y':          y_col,
            'x':          x_col,
            'beta_ols':   round(beta_ols, 5),
            'beta_tls':   round(beta_tls, 5),
            'adf_p_ols':  round(adf_ols['adf_pval'], 5),
            'adf_p_tls':  round(adf_tls['adf_pval'], 5),
            'hl_ols':     round(hl_ols, 1) if np.isfinite(hl_ols) else 9999,
            'hl_tls':     round(hl_tls, 1) if np.isfinite(hl_tls) else 9999,
            'hurst':      round(hu, 4) if not np.isnan(hu) else np.nan,
            'spread_std': round(best_spread.std(), 4),
            'tls_better': better_tls,
        })

pair_df = pd.DataFrame(pair_results)

# ── Pick the better method per row, build canonical fields ─────────────────
pair_df['best_method'] = np.where(pair_df['tls_better'], 'TLS', 'OLS')
pair_df['best_beta']   = np.where(pair_df['tls_better'], pair_df['beta_tls'], pair_df['beta_ols'])
pair_df['best_adf_p']  = np.minimum(pair_df['adf_p_ols'], pair_df['adf_p_tls'])
pair_df['best_hl']     = np.where(pair_df['tls_better'], pair_df['hl_tls'], pair_df['hl_ols'])

# ── Legacy column aliases so downstream cells (8, 10, 13-24) keep working ──
pair_df['adf_pval']    = pair_df['best_adf_p']
pair_df['hedge_beta']  = pair_df['best_beta']
pair_df['half_life']   = pair_df['best_hl']

# Composite score: low ADF p + short HL + low Hurst → high rank
pair_df['score'] = (
    pair_df['best_adf_p'].rank()              * 0.4 +
    pair_df['best_hl'].clip(upper=5000).rank() * 0.4 +
    pair_df['hurst'].rank()                   * 0.2
)
pair_df = pair_df.sort_values('score').reset_index(drop=True)

print(f'Total pairs tested: {len(pair_df)}')
print(f'TLS beats OLS on:   {pair_df["tls_better"].sum()}/{len(pair_df)} pairs')
print(f'Stationary at 5%:   {(pair_df["best_adf_p"] < 0.05).sum()} pairs')
print(f'Stationary at 1%:   {(pair_df["best_adf_p"] < 0.01).sum()} pairs')
print('\n=== TOP 20 PAIRS — OLS vs TLS comparison ===')
display(pair_df.head(20)[['category','y','x','best_method','best_beta',
                           'adf_p_ols','adf_p_tls','best_hl','hurst','spread_std']])

In [ ]:
# %%
# ─── CELL 7b: TLS vs OLS — Where Orthogonal Regression Earns Alpha ──────────
#
# Visualize side-by-side OLS and TLS spreads for the top-improving pairs.
# Lower ADF p-value on TLS = TLS is the right hedge ratio for trading.
#
# ALPHA NOTE: Use the TLS β for the Trader skeleton on these pairs. The
# OLS spread embeds asymmetric noise that hides true mean reversion.

pair_df['adf_improvement'] = pair_df['adf_p_ols'] - pair_df['adf_p_tls']
tls_winners = pair_df[pair_df['tls_better']].nlargest(6, 'adf_improvement')

print('=== Pairs where TLS materially improves stationarity ===')
display(tls_winners[['category','y','x','beta_ols','beta_tls',
                      'adf_p_ols','adf_p_tls','adf_improvement']])

if len(tls_winners) > 0:
    nrows = min(len(tls_winners), 6)
    fig, axes = plt.subplots(nrows, 2, figsize=(18, 3.2 * nrows))
    if nrows == 1:
        axes = axes.reshape(1, 2)

    for i, (_, row) in enumerate(tls_winners.iterrows()):
        if i >= nrows: break
        y_arr = prices[row['y']].dropna().values
        x_arr = prices[row['x']].dropna().values
        n = min(len(y_arr), len(x_arr))
        y_arr, x_arr = y_arr[:n], x_arr[:n]

        _, _, sp_ols = ols_hedge_ratio(y_arr, x_arr)
        _, _, sp_tls = tls_hedge_ratio(y_arr, x_arr)

        axes[i, 0].plot(sp_ols, lw=0.5, color='steelblue')
        axes[i, 0].axhline(sp_ols.mean(), color='black', ls='--', lw=1)
        axes[i, 0].set_title(
            f"OLS  {row['y'].split('_')[-1]}/{row['x'].split('_')[-1]}  "
            f"β={row['beta_ols']:.3f}  ADFp={row['adf_p_ols']:.4f}",
            fontsize=9, fontweight='bold')

        axes[i, 1].plot(sp_tls, lw=0.5, color='darkgreen')
        axes[i, 1].axhline(sp_tls.mean(), color='black', ls='--', lw=1)
        axes[i, 1].set_title(
            f"TLS  {row['y'].split('_')[-1]}/{row['x'].split('_')[-1]}  "
            f"β={row['beta_tls']:.3f}  ADFp={row['adf_p_tls']:.4f}",
            fontsize=9, fontweight='bold')

    plt.suptitle('OLS vs TLS Spreads — Where Orthogonal Regression Wins',
                 fontsize=12, y=1.001)
    plt.tight_layout()
    plt.savefig('tls_vs_ols_comparison.png', bbox_inches='tight')
    plt.show()
else:
    print('OLS dominates on every pair — no TLS-only alpha available.')

In [ ]:
# %%
# ─── CELL 8: Best Pairs — Per Category Ranking ──────────────────────────────

print('=== Best pair per category (by ADF p-value) ===')
best_by_cat = pair_df.loc[pair_df.groupby('category')['adf_pval'].idxmin()]
display(best_by_cat[['category','y','x','hedge_beta','adf_pval','half_life','hurst','spread_std']])

print('\n=== All stationary pairs (adf_pval < 0.05) ===')
stationary_pairs = pair_df[pair_df['adf_pval'] < 0.05].copy()
display(stationary_pairs[['category','y','x','hedge_beta','adf_pval','half_life','hurst','spread_std']])

In [ ]:
# %%
# ─── CELL 9: Johansen Basket Test (All 5 Assets Per Category) ───────────────
#
# Johansen tests whether the FULL group of 5 assets is cointegrated.
# Reports the number of cointegrating vectors at 5% significance.

johansen_results = []

for cat, cols in CATEGORIES.items():
    cat_df = prices[cols].dropna()
    try:
        result = coint_johansen(cat_df, det_order=0, k_ar_diff=1)
        # Trace statistic vs 5% critical value
        trace_stat   = result.lr1          # shape (5,)
        crit_vals_5  = result.cvt[:, 1]    # 5% critical values
        n_coint = int((trace_stat > crit_vals_5).sum())
        johansen_results.append({
            'category':         cat,
            'n_coint_vectors':  n_coint,
            'trace_r0':         round(trace_stat[0], 2),
            'crit_r0_5pct':     round(crit_vals_5[0], 2),
            'trace_r1':         round(trace_stat[1], 2),
            'crit_r1_5pct':     round(crit_vals_5[1], 2),
        })
    except Exception as e:
        johansen_results.append({'category': cat, 'error': str(e)})

joh_df = pd.DataFrame(johansen_results).set_index('category')
print('=== Johansen Basket Cointegration (5 assets per category) ===')
print('n_coint_vectors = 0 → no cointegration; >0 → basket tradeable')
display(joh_df)

In [ ]:
# %%
# ─── CELL 10: Top Spreads — Visual Stationarity Check ───────────────────────
#
# Plot the best spread per category (from EG results) with:
#   - Rolling mean & ±1σ / ±2σ bands
#   - ADF / half-life annotation

top_pairs_per_cat = pair_df.groupby('category').apply(
    lambda g: g.nsmallest(1, 'adf_pval').iloc[0]
).reset_index(drop=True)

fig, axes = plt.subplots(5, 2, figsize=(20, 28))
axes = axes.flatten()

ROLL_WIN = 200  # rolling window for bands

for idx, row in top_pairs_per_cat.iterrows():
    ax = axes[idx]
    y  = prices[row['y']].dropna()
    x  = prices[row['x']].dropna()
    idx_common = y.index.intersection(x.index)
    y, x = y.loc[idx_common], x.loc[idx_common]

    _, spread = fit_hedge_ratio(y, x)

    roll_mean = spread.rolling(ROLL_WIN).mean()
    roll_std  = spread.rolling(ROLL_WIN).std()

    ax.plot(spread.values, lw=0.6, color='steelblue', label='Spread')
    ax.plot(roll_mean.values, lw=1.2, color='black', ls='--', label='Rolling mean')
    ax.fill_between(range(len(spread)),
                    (roll_mean - roll_std).values, (roll_mean + roll_std).values,
                    alpha=0.2, color='orange', label='±1σ')
    ax.fill_between(range(len(spread)),
                    (roll_mean - 2*roll_std).values, (roll_mean + 2*roll_std).values,
                    alpha=0.1, color='red', label='±2σ')

    short_y = row['y'].split('_')[-1]
    short_x = row['x'].split('_')[-1]
    hl_str  = f'{row["half_life"]:.0f}' if row['half_life'] < 9000 else '∞'
    ax.set_title(
        f"{row['category']}: {short_y}−{row['hedge_beta']:.3f}×{short_x}\n"
        f"ADF p={row['adf_pval']:.4f}  HL={hl_str}  H={row['hurst']:.3f}",
        fontsize=9, fontweight='bold'
    )
    ax.legend(fontsize=6)
    ax.set_xlabel('Time step')

plt.suptitle('Best Cointegrated Spread Per Category (rolling ±1σ / ±2σ bands)', 
             fontsize=13, y=1.001)
plt.tight_layout()
plt.savefig('top_spreads.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%
# ─── CELL 11: Correlation Heatmaps Per Category ──────────────────────────────

fig, axes = plt.subplots(3, 4, figsize=(22, 16))
axes = axes.flatten()

for idx, (cat, cols) in enumerate(CATEGORIES.items()):
    ax = axes[idx]
    corr = prices[cols].corr()
    # Shorten labels for readability
    short_cols = [c.split('_')[-1] for c in cols]
    corr.index = short_cols
    corr.columns = short_cols
    sns.heatmap(corr, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=-1, vmax=1, center=0, square=True, cbar=False,
                annot_kws={'size': 8})
    ax.set_title(f'{cat}', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelsize=7, rotation=45)
    ax.tick_params(axis='y', labelsize=7, rotation=0)

# Hide unused axes
for i in range(len(CATEGORIES), len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Correlation Heatmaps — All 10 Categories', fontsize=14, y=1.002)
plt.tight_layout()
plt.savefig('correlation_heatmaps.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%
# ─── CELL 12: Lead-Lag Analysis — Vectorized Cross-Correlation by Category ──
#
# For each pair within a category, compute cross-correlation of RETURNS
# at lags -30 to +30 ticks. A peak at lag k ≠ 0 means one asset leads
# the other by k ticks — exploitable for directional signals.
#
# This version vectorizes correlation across all within-category pairs for
# each lag, avoiding the pair loop and repeated scalar correlation calls.

MAX_LAG = 30

fig, axes = plt.subplots(5, 2, figsize=(20, 28))
axes = axes.flatten()

lead_lag_results = []
lags_arr = np.arange(-MAX_LAG, MAX_LAG + 1)


def _column_corr(a, b):
    """Correlation for matching columns in two 2-D arrays."""
    a = a - a.mean(axis=0, keepdims=True)
    b = b - b.mean(axis=0, keepdims=True)
    denom = np.sqrt((a * a).sum(axis=0) * (b * b).sum(axis=0))
    return np.divide((a * b).sum(axis=0), denom, out=np.full(a.shape[1], np.nan), where=denom > 0)


for idx, (cat, cols) in enumerate(CATEGORIES.items()):
    ax = axes[idx]
    cat_rets = prices[cols].pct_change().dropna()
    ret_arr = cat_rets.to_numpy(dtype=float)
    n = ret_arr.shape[0]

    pair_idx = np.array(list(itertools.combinations(range(len(cols)), 2)))
    y_idx = pair_idx[:, 0]
    x_idx = pair_idx[:, 1]
    pair_labels = [(cols[i], cols[j]) for i, j in pair_idx]

    xcorr_mat = np.full((len(lags_arr), len(pair_labels)), np.nan)
    for lag_i, lag in enumerate(lags_arr):
        if lag >= 0:
            if n <= lag:
                continue
            y_seg = ret_arr[lag:, y_idx]
            x_seg = ret_arr[:n-lag, x_idx]
        else:
            lead = -lag
            if n <= lead:
                continue
            y_seg = ret_arr[:n-lead, y_idx]
            x_seg = ret_arr[lead:, x_idx]
        xcorr_mat[lag_i] = _column_corr(y_seg, x_seg)

    peak_abs_idx = np.nanargmax(np.abs(xcorr_mat), axis=0)
    peak_vals = xcorr_mat[peak_abs_idx, np.arange(len(pair_labels))]
    peak_lags = lags_arr[peak_abs_idx]

    for pair_i, (y_col, x_col) in enumerate(pair_labels):
        lead_lag_results.append({
            'category': cat, 'y': y_col, 'x': x_col,
            'peak_xcorr': round(peak_vals[pair_i], 4),
            'peak_lag':   int(peak_lags[pair_i]),
            'xcorr_lag0': round(xcorr_mat[MAX_LAG, pair_i], 4),
        })

    off_zero = np.abs(xcorr_mat).copy()
    off_zero[MAX_LAG, :] = 0
    best_flat = np.nanargmax(off_zero)
    best_lag_i, best_pair_i = np.unravel_index(best_flat, off_zero.shape)
    y_col, x_col = pair_labels[best_pair_i]
    xcorrs = xcorr_mat[:, best_pair_i]

    sy = y_col.split('_')[-1]
    sx = x_col.split('_')[-1]
    ax.bar(lags_arr, xcorrs, color=['tomato' if l == 0 else 'steelblue'
                                     for l in lags_arr], width=0.8, alpha=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(0, color='red', lw=1, ls='--', alpha=0.5)
    conf = 1.96 / np.sqrt(len(cat_rets))
    ax.axhline(conf, color='green', lw=0.8, ls=':', label=f'95% CI')
    ax.axhline(-conf, color='green', lw=0.8, ls=':')
    ax.set_title(f'{cat}\nBest lead-lag: {sy} vs {sx}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Lag (+ = y leads x)')
    ax.set_ylabel('Cross-corr')
    ax.legend(fontsize=7)

plt.suptitle('Return Cross-Correlations (lags −30 to +30) — Best Pair Per Category',
             fontsize=13, y=1.001)
plt.tight_layout()
plt.savefig('lead_lag_analysis.png', bbox_inches='tight')
plt.show()

ll_df = pd.DataFrame(lead_lag_results).sort_values('peak_lag', key=abs, ascending=False)
print('\n=== Strongest off-zero lead-lag relationships ===')
display(ll_df[ll_df['peak_lag'] != 0].head(20))


In [ ]:
# %%
# ─── CELL 13: STEP 2 — Pair Trading Signal Engine ───────────────────────────
#
# For each candidate pair (from pair_df):
#   1. Compute a rolling z-score of the OLS spread
#   2. Generate entry/exit signals
#   3. Enforce position limit of ±10 on EACH leg
#
# Signal logic:
#   z > +entry_z  → SHORT spread  (sell Y, buy X scaled by β)
#   z < -entry_z  → LONG spread   (buy Y, sell X scaled by β)
#   |z| < exit_z  → CLOSE         (flatten)
#
# Position sizing: with limit=10 on each leg, we cap at qty = min(10, floor(10/β))
# (when β ≈ 1 this is just ±10 on both legs)

ENTRY_Z  = 1.5    # z-score threshold to enter
EXIT_Z   = 0.3    # z-score threshold to exit
ROLL_WIN = 300    # rolling window for z-score computation
TRADE_QTY = 5     # units per leg (conservative; max 10)


def compute_zscore(spread: pd.Series, window: int) -> pd.Series:
    mu  = spread.rolling(window).mean()
    sig = spread.rolling(window).std()
    return (spread - mu) / sig


def generate_signals(zscore: pd.Series,
                     entry_z: float = ENTRY_Z,
                     exit_z:  float = EXIT_Z) -> pd.Series:
    """
    Returns a position series: +1 = long spread, -1 = short spread, 0 = flat.
    Holds position until exit threshold is crossed (state machine).
    """
    pos = pd.Series(0, index=zscore.index, dtype=int)
    current = 0
    for i, z in enumerate(zscore):
        if np.isnan(z):
            pos.iloc[i] = 0
            continue
        if current == 0:
            if z >  entry_z: current = -1   # short spread
            if z < -entry_z: current =  1   # long spread
        elif current ==  1 and z > -exit_z: current = 0
        elif current == -1 and z <  exit_z: current = 0
        pos.iloc[i] = current
    return pos


def backtest_pair(y: pd.Series, x: pd.Series,
                  cat: str, y_name: str, x_name: str,
                  roll_win: int = ROLL_WIN,
                  entry_z: float = ENTRY_Z,
                  exit_z:  float = EXIT_Z,
                  qty: int = TRADE_QTY) -> dict:
    """Simple mark-to-market backtest of a single pair. No transaction costs."""
    common = y.index.intersection(x.index)
    y, x   = y.loc[common], x.loc[common]

    beta, spread = fit_hedge_ratio(y, x)
    zscore = compute_zscore(spread, roll_win)
    pos    = generate_signals(zscore, entry_z, exit_z)

    # PnL: long spread → long Y, short X*β
    y_ret = y.diff()
    x_ret = x.diff()
    pnl   = pos.shift(1) * (y_ret - beta * x_ret) * qty
    cum   = pnl.cumsum()

    total_trades = (pos.diff().abs() > 0).sum()
    return {
        'category':   cat,
        'y':          y_name,
        'x':          x_name,
        'beta':       round(beta, 5),
        'total_pnl':  round(cum.iloc[-1], 2),
        'sharpe':     round(pnl.mean() / pnl.std() * np.sqrt(252 * 1000), 3) if pnl.std() > 0 else np.nan,
        'max_dd':     round((cum - cum.cummax()).min(), 2),
        'n_trades':   int(total_trades),
        'pnl_series': cum,
        'pos_series': pos,
        'zscore':     zscore,
        'spread':     spread,
    }


print('Signal engine defined. Running backtests on all stationary pairs...')

# Run on all pairs with adf_pval < 0.1
candidate_pairs = pair_df[pair_df['adf_pval'] < 0.10].copy()
print(f'Candidate pairs to backtest: {len(candidate_pairs)}')

bt_results = []
bt_detail  = {}  # store full result for plotting top pairs

for _, row in candidate_pairs.iterrows():
    res = backtest_pair(
        prices[row['y']], prices[row['x']],
        row['category'], row['y'], row['x']
    )
    key = (row['category'], row['y'], row['x'])
    bt_detail[key] = res
    bt_results.append({k: v for k, v in res.items()
                       if k not in ('pnl_series', 'pos_series', 'zscore', 'spread')})

bt_df = pd.DataFrame(bt_results).sort_values('total_pnl', ascending=False)
print('\n=== Backtest Results (sorted by total PnL, no transaction costs) ===')
display(bt_df.head(20))

In [ ]:
# %%
# ─── CELL 14: Backtest Plots — Top 6 Pairs ──────────────────────────────────

top6 = bt_df.head(6)

fig, axes = plt.subplots(6, 3, figsize=(22, 30))

for row_idx, (_, row) in enumerate(top6.iterrows()):
    key    = (row['category'], row['y'], row['x'])
    detail = bt_detail[key]

    ax_z  = axes[row_idx, 0]
    ax_p  = axes[row_idx, 1]
    ax_pnl= axes[row_idx, 2]

    z   = detail['zscore']
    pos = detail['pos_series']
    pnl = detail['pnl_series']

    # -- Z-score with entry/exit bands --
    ax_z.plot(z.values, lw=0.5, color='steelblue')
    ax_z.axhline(ENTRY_Z,  color='red',   lw=0.8, ls='--', label=f'+{ENTRY_Z}')
    ax_z.axhline(-ENTRY_Z, color='green', lw=0.8, ls='--', label=f'-{ENTRY_Z}')
    ax_z.axhline(0, color='black', lw=0.6)
    ax_z.set_title(f"{row['category']}\n{row['y'].split('_')[-1]} / {row['x'].split('_')[-1]}\nZ-score",
                   fontsize=8)
    ax_z.legend(fontsize=6)
    ax_z.set_ylim(-5, 5)

    # -- Position & spread --
    ax_p2 = ax_p.twinx()
    ax_p.plot(detail['spread'].values, lw=0.5, color='gray', alpha=0.5, label='Spread')
    ax_p2.step(range(len(pos)), pos.values, lw=0.8, color='darkorange', label='Position')
    ax_p.set_title('Spread & Position', fontsize=8)
    ax_p.set_ylabel('Spread', fontsize=7)
    ax_p2.set_ylabel('Position', fontsize=7)

    # -- Cumulative PnL --
    ax_pnl.plot(pnl.values, lw=0.8, color='darkgreen')
    ax_pnl.fill_between(range(len(pnl)), pnl.values, alpha=0.15, color='green')
    ax_pnl.set_title(f'Cum PnL={row["total_pnl"]:.0f}  Sharpe={row["sharpe"]:.2f}\nMaxDD={row["max_dd"]:.0f}  Trades={row["n_trades"]}',
                     fontsize=8)
    ax_pnl.axhline(0, color='black', lw=0.6)

plt.suptitle(f'Top 6 Pairs — Z-score / Spread / Cumulative PnL (entry={ENTRY_Z}, exit={EXIT_Z}, qty={TRADE_QTY})',
             fontsize=12, y=1.001)
plt.tight_layout()
plt.savefig('top_pairs_backtest.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%
# ─── CELL 16: VECTORIZED Rolling Hedge Ratio (closed-form OLS + TLS) ─────
#
# Replaces the O(N·W) Python loop with vectorized rolling sufficient
# statistics. OLS uses the closed-form one-regressor formula per window:
#
#     β = cov(x, y) / var(x)
#     α = mean(y) − β·mean(x)
#
# No matrix inversion, no batched linear solve, no least-squares solve.
# This avoids materializing the full (M, W, 2) design tensor while keeping
# the rolling calculation in NumPy array operations.
#
# ALPHA NOTE: Fast rolling β unlocks DYNAMIC HEDGE RATIOS in production.
# Static β is only correct on average; rolling β tracks regime shifts
# and reduces basis risk. In the Trader, use the most recent rolling β
# at trade-open time (recompute every 50–200 ticks).

def rolling_ols_vectorized(y, x, window):
    """
    Vectorized rolling OLS for the simple model y = α + β·x.
    Uses closed-form sufficient statistics per window.

    Returns:
      betas, alphas — each (N − window + 1,), aligned to window-end.
    """
    y = np.asarray(y, dtype=float).ravel()
    x = np.asarray(x, dtype=float).ravel()
    n = len(y)
    if n < window:
        return np.full(0, np.nan), np.full(0, np.nan)

    yw = sliding_window_view(y, window)        # (M, W) view, no copy
    xw = sliding_window_view(x, window)        # (M, W) view, no copy

    x_mean = xw.mean(axis=1)
    y_mean = yw.mean(axis=1)
    x_cent = xw - x_mean[:, None]
    y_cent = yw - y_mean[:, None]
    denom = (x_cent * x_cent).sum(axis=1)
    beta = np.divide((x_cent * y_cent).sum(axis=1), denom,
                     out=np.full_like(denom, np.nan, dtype=float), where=denom > 1e-12)
    alpha = y_mean - beta * x_mean
    return beta, alpha


def rolling_tls_vectorized(y, x, window):
    """
    Vectorized rolling TLS via the closed-form 2×2 eigenvector of the
    windowed covariance matrix.
    For the 2×2 symmetric matrix [[Sxx, Sxy], [Sxy, Syy]] the smaller
    eigenvalue is λ_min = ((Sxx+Syy) − √((Sxx−Syy)² + 4·Sxy²)) / 2 and
    its eigenvector gives the orthogonal-residual direction.
    """
    y = np.asarray(y, dtype=float).ravel()
    x = np.asarray(x, dtype=float).ravel()
    yw = sliding_window_view(y, window)
    xw = sliding_window_view(x, window)

    yc = yw - yw.mean(axis=1, keepdims=True)
    xc = xw - xw.mean(axis=1, keepdims=True)

    sxx = (xc * xc).sum(axis=1)
    syy = (yc * yc).sum(axis=1)
    sxy = (xc * yc).sum(axis=1)

    disc    = np.sqrt((sxx - syy) ** 2 + 4 * sxy ** 2)
    lam_min = 0.5 * ((sxx + syy) - disc)
    denom   = lam_min - sxx
    return np.where(np.abs(denom) > 1e-12, -sxy / denom, np.nan)


# ── Demonstration on best pair per category ───────────────────────────────
ROLL_BETA_WIN = 2000

best_pairs_per_cat = (
    pair_df.groupby('category')
            .apply(lambda g: g.nsmallest(1, 'best_adf_p').iloc[0])
            .reset_index(drop=True)
)

import time

fig, axes = plt.subplots(5, 2, figsize=(18, 22))
axes = axes.flatten()

beta_stability = []   # collected for the structural-break cell

for idx, (_, row) in enumerate(best_pairs_per_cat.iterrows()):
    ax = axes[idx]
    y_full = prices[row['y']].dropna()
    x_full = prices[row['x']].dropna()
    common = y_full.index.intersection(x_full.index)
    y, x   = y_full.loc[common].values, x_full.loc[common].values

    t0 = time.time()
    betas_ols, _ = rolling_ols_vectorized(y, x, ROLL_BETA_WIN)
    betas_tls    = rolling_tls_vectorized(y, x, ROLL_BETA_WIN)
    elapsed = time.time() - t0

    cv_ols = betas_ols.std() / abs(betas_ols.mean()) if betas_ols.mean() else np.nan
    valid_tls = betas_tls[~np.isnan(betas_tls)]
    cv_tls = valid_tls.std() / abs(valid_tls.mean()) if len(valid_tls) and valid_tls.mean() else np.nan

    beta_stability.append({
        'category':    row['category'],
        'y':           row['y'],
        'x':           row['x'],
        'beta_mean':   round(betas_ols.mean(), 4),
        'beta_std':    round(betas_ols.std(),  4),
        'cv_ols':      round(cv_ols, 4),
        'cv_tls':      round(cv_tls, 4) if not np.isnan(cv_tls) else np.nan,
        'beta_first':  round(betas_ols[:200].mean(), 4),
        'beta_last':   round(betas_ols[-200:].mean(), 4),
        'compute_ms':  round(elapsed * 1000, 2),
    })

    ax.plot(betas_ols, lw=0.7, color='steelblue', label='Rolling OLS β')
    ax.plot(betas_tls, lw=0.7, color='darkgreen', label='Rolling TLS β', alpha=0.7)
    ax.axhline(betas_ols.mean(), color='black', lw=1, ls='--')
    ax.fill_between(np.arange(len(betas_ols)),
                    betas_ols.mean() - betas_ols.std(),
                    betas_ols.mean() + betas_ols.std(),
                    alpha=0.12, color='steelblue')
    ax.set_title(
        f"{row['category']}: rolling β  "
        f"{row['y'].split('_')[-1]}/{row['x'].split('_')[-1]}\n"
        f"CV_OLS={cv_ols:.3f}  CV_TLS={cv_tls:.3f}  ({elapsed*1000:.1f}ms)",
        fontsize=8, fontweight='bold')
    ax.legend(fontsize=6)
    ax.set_xlabel('Time step (window-end-aligned)')
    ax.set_ylabel('Hedge ratio β')

plt.suptitle(f'Vectorized Rolling β (window={ROLL_BETA_WIN}) — OLS + TLS',
             fontsize=12, y=1.001)
plt.tight_layout()
plt.savefig('rolling_beta_vectorized.png', bbox_inches='tight')
plt.show()

beta_stab_df = pd.DataFrame(beta_stability)
print('\n=== Rolling-β stability metrics (per-category best pair) ===')
display(beta_stab_df)

In [ ]:
# %%
# ─── CELL 16b: Structural Break Detection — Day-over-Day β Drift ────────────
#
# Two complementary stability tests for every top pair:
#   (1) DAY-OVER-DAY DRIFT:  |β_dayN − β_day1| / |β_day1|  > 20%   → fail
#   (2) ROLLING-β CV:        std(β_roll) / |mean(β_roll)| > 30%    → fail
#
# Decision tree:
#   STABLE ✓   passes both        → trade with static β
#   WATCH      fails one of two   → trade with rolling β (re-estimate)
#   BREAK ⚠    fails both         → DO NOT TRADE; relationship breaking
#
# ALPHA NOTE: A pair that passes (1) but fails (2) is intra-day noisy but
# day-anchored — these are GOOD targets if you re-estimate β each tick.
# A pair that fails (1) but passes (2) has a regime shift between days —
# DANGEROUS in production; the next live day is unknown territory.

DRIFT_THRESHOLD = 0.20      # 20% relative β change Day 1 → Day 3
CV_THRESHOLD    = 0.30      # 30% rolling-β coefficient of variation
DAY_WIN         = 8000      # ~80% of a single day (10K ticks/day)


def structural_break_check(y_name, x_name, day_slices_dict, days_list):
    """Per-day β + drift + rolling-β CV."""
    y_full = prices[y_name].dropna()
    x_full = prices[x_name].dropna()
    common = y_full.index.intersection(x_full.index)
    y, x = y_full.loc[common].values, x_full.loc[common].values

    # ── Per-day β (static OLS on each day's slice) ──
    day_betas = {}
    for d in days_list:
        y_d = day_slices_dict[d][y_name].dropna()
        x_d = day_slices_dict[d][x_name].dropna()
        cmn = y_d.index.intersection(x_d.index)
        if len(cmn) < 200:
            day_betas[d] = np.nan
            continue
        _, b, _ = ols_hedge_ratio(y_d.loc[cmn].values, x_d.loc[cmn].values)
        day_betas[d] = b

    b_first, b_last = day_betas[days_list[0]], day_betas[days_list[-1]]
    rel_drift = abs(b_last - b_first) / abs(b_first) if b_first else np.nan

    # ── Rolling-β CV (uses our vectorized function) ──
    if len(y) >= DAY_WIN:
        betas_roll, _ = rolling_ols_vectorized(y, x, DAY_WIN)
        cv = betas_roll.std() / abs(betas_roll.mean()) if betas_roll.mean() else np.nan
    else:
        cv = np.nan

    return {
        f'beta_d{days_list[0]}': round(day_betas[days_list[0]], 4) if not np.isnan(day_betas[days_list[0]]) else np.nan,
        f'beta_d{days_list[1]}': round(day_betas[days_list[1]], 4) if not np.isnan(day_betas[days_list[1]]) else np.nan,
        f'beta_d{days_list[2]}': round(day_betas[days_list[2]], 4) if not np.isnan(day_betas[days_list[2]]) else np.nan,
        'rel_drift':  round(rel_drift, 4) if not np.isnan(rel_drift) else np.nan,
        'rolling_cv': round(cv, 4) if not np.isnan(cv) else np.nan,
        'drift_fail': bool(rel_drift > DRIFT_THRESHOLD) if not np.isnan(rel_drift) else False,
        'cv_fail':    bool(cv > CV_THRESHOLD) if not np.isnan(cv) else False,
    }


# ── Run on top 25 pairs by best ADF p-value ──
top25 = pair_df.nsmallest(25, 'best_adf_p')
break_rows = []
for _, row in top25.iterrows():
    res = structural_break_check(row['y'], row['x'], day_slices, DAYS)
    break_rows.append({'category': row['category'], 'y': row['y'], 'x': row['x'],
                        'best_adf_p': row['best_adf_p'], **res})

break_df = pd.DataFrame(break_rows)
break_df['structural_status'] = np.where(
    break_df['drift_fail'] & break_df['cv_fail'],   'BREAK',
    np.where(break_df['drift_fail'] | break_df['cv_fail'], 'WATCH', 'STABLE')
)

n_stable = int((break_df['structural_status'] == 'STABLE').sum())
n_watch  = int((break_df['structural_status'] == 'WATCH').sum())
n_break  = int((break_df['structural_status'] == 'BREAK').sum())

print('=== Structural Stability — Top 25 Pairs ===')
print(f'STABLE (trade with static β):     {n_stable}')
print(f'WATCH  (use rolling β):           {n_watch}')
print(f'BREAK  (DO NOT TRADE):            {n_break}')
display(break_df.sort_values(['structural_status', 'best_adf_p']))

In [ ]:
# %%
# ─── CELL 16c: Bid-Ask Spread Diagnostics — Per-Asset Friction ──────────────
#
# bid_pivot and ask_pivot were already created in Cell 2.
# Friction per asset = ask_1 − bid_1.  This is the cost of crossing
# the book once. A round-trip pair trade pays this 4 times in total
# (enter Y, enter X, exit Y, exit X) ≈ 2 × (bay + |β| · bax).
#
# ALPHA NOTE: Assets with mean BA spread > spread_std of the candidate
# pair cannot be profitably traded — there is no edge to capture.
# Filter early: any pair where ba_y > spread_std is dead-on-arrival.

bid_ask_spread = (asks - bids)

ba_stats = pd.DataFrame({
    'mean_ba':  bid_ask_spread.mean(),
    'std_ba':   bid_ask_spread.std(),
    'p95_ba':   bid_ask_spread.quantile(0.95),
    'max_ba':   bid_ask_spread.max(),
}).round(3)
ba_stats = ba_stats.sort_values('mean_ba')

print('=== Per-asset bid-ask spread (lower = cheaper to trade) ===')
display(ba_stats)

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(13, 11))
ba_stats['mean_ba'].plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Mean Bid-Ask Spread Per Asset (sorted ascending)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('mean(ask₁ − bid₁)')
plt.tight_layout()
plt.savefig('bid_ask_spread_per_asset.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%
# ─── CELL 16d: Friction Ratio — The True Tradability Test ───────────────────
#
# For every candidate pair compute:
#
#     theoretical_edge  =  entry_z · spread_std            (price units)
#     round_trip_cost   =  2 · ( ba_y + |β| · ba_x )       (4 book-crosses)
#     friction_ratio    =  round_trip_cost / theoretical_edge
#
# A friction_ratio > 0.40 ⇒ costs eat >40% of the edge ⇒ REJECT the pair.
# We default entry_z = 1.5; widen it (e.g. 2.0) to push more pairs into
# tradable territory at the cost of fewer signals.
#
# ALPHA NOTE: This is the SINGLE MOST IMPORTANT filter. Most "stationary"
# pairs from a mid-quote backtest die here — they only looked profitable
# because the simulation never paid the bid-ask. The pairs that survive
# this filter are the REAL alpha you can deploy live.

ENTRY_Z_FOR_EDGE = 1.5
MAX_FRICTION     = 0.40

# Friction-test the top 30 pairs by best ADF p-value
top_for_friction = pair_df.nsmallest(30, 'best_adf_p').copy()

friction_rows = []
for _, row in top_for_friction.iterrows():
    y_name, x_name = row['y'], row['x']
    beta = row['best_beta']

    ba_y = ba_stats.loc[y_name, 'mean_ba']
    ba_x = ba_stats.loc[x_name, 'mean_ba']

    # Recompute spread on the better hedge ratio
    y_arr = prices[y_name].dropna().values
    x_arr = prices[x_name].dropna().values
    n = min(len(y_arr), len(x_arr))
    spread = y_arr[:n] - beta * x_arr[:n]
    spread_std = spread.std()

    edge_price = ENTRY_Z_FOR_EDGE * spread_std
    round_trip = 2.0 * (ba_y + abs(beta) * ba_x)
    friction   = round_trip / edge_price if edge_price > 0 else np.inf

    friction_rows.append({
        'category':   row['category'],
        'y':          y_name,
        'x':          x_name,
        'method':     row['best_method'],
        'beta':       beta,
        'best_adf_p': row['best_adf_p'],
        'spread_std': round(spread_std, 4),
        'edge':       round(edge_price, 3),
        'ba_y':       round(ba_y, 3),
        'ba_x':       round(ba_x, 3),
        'round_trip': round(round_trip, 3),
        'friction':   round(friction, 4),
        'tradable':   bool(friction <= MAX_FRICTION),
    })

friction_df = pd.DataFrame(friction_rows).sort_values('friction')

print(f'=== Friction Ratio — Top 30 Cointegrated Pairs ===')
print(f'Threshold: round-trip cost ≤ {MAX_FRICTION*100:.0f}% of {ENTRY_Z_FOR_EDGE}σ edge')
print(f'TRADABLE  pairs:   {int(friction_df["tradable"].sum())}')
print(f'REJECTED pairs:    {int((~friction_df["tradable"]).sum())}')
display(friction_df)

# ── COMBINED FILTER: friction OK + structurally stable ─────────────────────
final_tradable = friction_df[friction_df['tradable']].merge(
    break_df[['category', 'y', 'x', 'structural_status', 'rel_drift', 'rolling_cv']],
    on=['category', 'y', 'x'], how='left'
)
final_tradable = final_tradable[
    final_tradable['structural_status'].isin(['STABLE', 'WATCH'])
].sort_values('best_adf_p').reset_index(drop=True)

print('\n╔═══════════════════════════════════════════════════════════════════╗')
print('║  FINAL TRADABLE PAIRS                                             ║')
print('║  (low ADF p + friction ≤ 40% + structurally stable)               ║')
print('╚═══════════════════════════════════════════════════════════════════╝')
display(final_tradable[['category','y','x','method','beta','best_adf_p',
                         'spread_std','friction','structural_status',
                         'rel_drift','rolling_cv']])

# Save for downstream cells
final_tradable.to_csv('final_tradable_pairs.csv', index=False)
print(f'\nSaved {len(final_tradable)} final tradable pairs to final_tradable_pairs.csv')

In [ ]:
# %%
# ─── CELL 16: Rolling-Window Stability Check ────────────────────────────────
#
# Is the cointegration relationship stable across the 3 days?
# Rolling hedge ratio β tells us if the pair relationship drifts.
# A stable β = robust pair. A drifting β = avoid or use dynamic re-estimation.

ROLL_BETA_WIN = 2000  # ticks for rolling OLS

fig, axes = plt.subplots(5, 2, figsize=(18, 22))
axes = axes.flatten()

for idx, (_, row) in enumerate(best_pairs_per_cat.iterrows()):
    ax = axes[idx]
    y = prices[row['y']].dropna()
    x = prices[row['x']].dropna()
    common = y.index.intersection(x.index)
    y, x = y.loc[common], x.loc[common]

    beta_vals, _ = rolling_ols_vectorized(y.values, x.values, ROLL_BETA_WIN)
    betas = pd.Series(np.nan, index=y.index)
    if len(beta_vals):
        betas.iloc[ROLL_BETA_WIN - 1:] = beta_vals

    coeff_var = betas.std() / betas.mean() if betas.mean() != 0 else np.nan

    ax.plot(betas.values, lw=0.7, color='purple')
    ax.axhline(betas.mean(), color='black', lw=1, ls='--', label=f'Mean β={betas.mean():.3f}')
    ax.fill_between(range(len(betas)),
                    betas.mean() - betas.std(), betas.mean() + betas.std(),
                    alpha=0.15, color='purple')
    ax.set_title(
        f"{row['category']}: Rolling β  "
        f"{row['y'].split('_')[-1]}/{row['x'].split('_')[-1]}\n"
        f"CV={coeff_var:.3f}  (low CV = stable pair)",
        fontsize=8)
    ax.legend(fontsize=7)
    ax.set_xlabel('Time step')
    ax.set_ylabel('Hedge ratio β')

plt.suptitle(f'Rolling Hedge Ratio Stability (window={ROLL_BETA_WIN} ticks)', fontsize=13, y=1.001)
plt.tight_layout()
plt.savefig('rolling_beta_stability.png', bbox_inches='tight')
plt.show()


In [ ]:
# %%
# ─── CELL 17: Special Analysis — Panels (Area Ratios) ───────────────────────
#
# PANEL_2X2 and PANEL_1X4 both have area 4 — if priced per unit area,
# they should be cointegrated with β ≈ 1.
# PANEL_4X4 area = 16, PANEL_2X4 = 8 → ratio 2:1 → test β ≈ 0.5.

panels = CATEGORIES['Panels']
panel_df = prices[panels].dropna()

print('=== Panels: Price level comparison ===')
print(panel_df.describe().round(2))

# Test same-area pair
_, sp_2x2_1x4 = fit_hedge_ratio(panel_df['PANEL_2X2'], panel_df['PANEL_1X4'])
adf_same_area = adf_test(sp_2x2_1x4)
print(f'\nPANEL_2X2 vs PANEL_1X4 (same area=4):')
print(f'  ADF p-value: {adf_same_area["adf_pval"]:.6f}  stationary={adf_same_area["stationary"]}')
print(f'  Half-life: {half_life(sp_2x2_1x4):.1f} ticks')

# Plot all panel price series and area-normalised prices
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Raw prices
for col in panels:
    axes[0].plot(panel_df[col].values, lw=0.6, label=col.replace('PANEL_', ''))
axes[0].set_title('Raw Panel Prices', fontweight='bold')
axes[0].legend(fontsize=7)

# Area-normalised (price / area)
for col in panels:
    area = PANEL_AREAS[col]
    axes[1].plot((panel_df[col] / area).values, lw=0.6, label=f'{col.replace("PANEL_","")} ÷{area}')
axes[1].set_title('Price Per Unit Area', fontweight='bold')
axes[1].legend(fontsize=7)

# Same-area spread
axes[2].plot(sp_2x2_1x4.values, lw=0.6, color='purple')
axes[2].axhline(sp_2x2_1x4.mean(), color='black', lw=1, ls='--')
axes[2].set_title(f'Spread: 2X2 − β×1X4\nADF p={adf_same_area["adf_pval"]:.4f}', fontweight='bold')

plt.tight_layout()
plt.savefig('panels_area_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%
# ─── CELL 22: Production Trader — Rolling TLS β + Z-score Pair Trading ──────
#
# Generates round5_trader.py — submission-ready, self-contained.
# The Trader maintains rolling price history per pair in traderData,
# recomputes TLS β every BETA_REFRESH ticks via closed-form 2×2 eigen,
# and trades z-score signals subject to ±10 position limit on each leg.
#
# CRITICAL FIXES baked in:
#   • int(...) wraps every Order price (V5/V6/V7 dashboard rejection bug)
#   • Position limit checked on BOTH legs before sending orders
#   • β jumps capped at 200% relative change (sanity-check vs spurious fits)
#   • Hedge X = round(β · ΔY) — exact pair hedging at every tick
#
# DEPLOYMENT WORKFLOW:
#   1. Cell 16d produces final_tradable_pairs.csv (filtered for friction)
#   2. Run Cell 22b to in-sample validate (rolling β + PnL)
#   3. Keep ONLY pairs with positive in-sample PnL → upload Cell 22's output

import os

# ── Step 1: Load tradable pairs from prior analysis ─────────────────────────
try:
    tradable_df = pd.read_csv('final_tradable_pairs.csv')
    print(f'Loaded {len(tradable_df)} candidate pairs from final_tradable_pairs.csv')
except FileNotFoundError:
    print('⚠️  final_tradable_pairs.csv not found — run Cell 16d first.')
    tradable_df = pd.DataFrame(columns=['y','x','beta'])

# Each tuple: (Y_product, X_product, β_init, entry_z, exit_z, qty_per_leg)
SELECTED_PAIRS = [
    (row['y'], row['x'], float(row['beta']), 1.5, 0.3, 5)
    for _, row in tradable_df.iterrows()
]
# Cap to 6 pairs to keep traderData small (JSON-serialised every tick)
SELECTED_PAIRS = SELECTED_PAIRS[:6]

print('\nSELECTED_PAIRS for Trader:')
for i, p in enumerate(SELECTED_PAIRS):
    print(f'  [{i}]  Y={p[0]:30}  X={p[1]:30}  β={p[2]:>+8.4f}  z=±{p[3]}  exit={p[4]}  qty={p[5]}')


# ── Step 2: Trader source template ──────────────────────────────────────────
TRADER_SRC = '''# IMC Prosperity 4 — Round 5 Trader (rolling TLS β pair trading)
# Generated by round5_analysis.ipynb Cell 22.
# Position limit: ±10 per product. All Order prices int()-wrapped.

import json, math
from typing import List
from datamodel import OrderDepth, TradingState, Order


# ─── PAIR CONFIG ────────────────────────────────────────────────────────
# Each tuple: (Y, X, beta_init, entry_z, exit_z, qty_per_leg)
SELECTED_PAIRS = __PAIRS__


# ─── HYPERPARAMETERS ───────────────────────────────────────────────────
POS_LIMIT          = 10      # IMC hard limit per product
BETA_WIN           = 500     # rolling β window (ticks)
Z_WIN              = 300     # z-score window
BETA_REFRESH       = 50      # recompute β every K ticks
MIN_HIST_FOR_TRADE = 150     # require this many ticks before any signal
MAX_BETA_JUMP_PCT  = 2.0     # reject β updates that flip by >200%


def tls_beta_2x2(y_list, x_list):
    """Closed-form TLS β via smallest-eigvec of 2×2 demeaned covariance."""
    n = len(x_list)
    if n < 30:
        return None
    mx = sum(x_list) / n
    my = sum(y_list) / n
    sxx = syy = sxy = 0.0
    for xi, yi in zip(x_list, y_list):
        dx, dy = xi - mx, yi - my
        sxx += dx*dx
        syy += dy*dy
        sxy += dx*dy
    disc    = math.sqrt((sxx - syy)**2 + 4*sxy*sxy)
    lam_min = 0.5 * ((sxx + syy) - disc)
    denom   = lam_min - sxx
    return -sxy / denom if abs(denom) > 1e-12 else None


def _mid(state, p):
    od = state.order_depths.get(p)
    if od is None or not od.buy_orders or not od.sell_orders:
        return None
    return (max(od.buy_orders) + min(od.sell_orders)) / 2.0


def _best_bid(state, p):
    od = state.order_depths.get(p)
    return int(max(od.buy_orders)) if od and od.buy_orders else None


def _best_ask(state, p):
    od = state.order_depths.get(p)
    return int(min(od.sell_orders)) if od and od.sell_orders else None


def _max_qty(base_qty, y_pos, x_pos, beta, direction):
    """Max ΔY that keeps both legs within ±POS_LIMIT after the trade.

    Long spread:  +ΔY on Y,  −β·ΔY on X   (sign='long', x_per_y = −β)
    Short spread: −ΔY on Y,  +β·ΔY on X   (sign='short', x_per_y = +β)
    """
    if direction == "long":
        cap_y   = POS_LIMIT - y_pos
        x_per_y = -beta
    else:
        cap_y   = POS_LIMIT + y_pos
        x_per_y = beta
    if abs(x_per_y) < 1e-9:
        cap_x = base_qty
    elif x_per_y > 0:
        cap_x = (POS_LIMIT - x_pos) / x_per_y
    else:
        cap_x = (POS_LIMIT + x_pos) / abs(x_per_y)
    return max(0, min(base_qty, int(cap_y), int(cap_x)))


class Trader:
    def run(self, state: TradingState):
        result = {}
        td = json.loads(state.traderData) if state.traderData else {}

        for (Y, X, beta_init, entry_z, exit_z, base_qty) in SELECTED_PAIRS:
            y_mid = _mid(state, Y)
            x_mid = _mid(state, X)
            if y_mid is None or x_mid is None:
                continue

            key = Y + "|" + X
            ps = td.get(key, {
                "y_hist": [], "x_hist": [], "sp_hist": [],
                "beta":   float(beta_init), "tick": 0,
            })

            # ── Update price history (capped at BETA_WIN) ──
            ps["y_hist"].append(y_mid)
            ps["x_hist"].append(x_mid)
            if len(ps["y_hist"]) > BETA_WIN:
                ps["y_hist"] = ps["y_hist"][-BETA_WIN:]
                ps["x_hist"] = ps["x_hist"][-BETA_WIN:]
            ps["tick"] += 1

            # ── Refresh β every K ticks (with sanity check) ──
            if ps["tick"] % BETA_REFRESH == 0 and len(ps["y_hist"]) >= 100:
                new_b = tls_beta_2x2(ps["y_hist"], ps["x_hist"])
                if new_b is not None:
                    rel_jump = abs(new_b - ps["beta"]) / max(abs(ps["beta"]), 1e-9)
                    if rel_jump < MAX_BETA_JUMP_PCT:
                        ps["beta"] = new_b

            beta = ps["beta"]

            # ── Spread + z-score ──
            sp_now = y_mid - beta * x_mid
            ps["sp_hist"].append(sp_now)
            if len(ps["sp_hist"]) > Z_WIN:
                ps["sp_hist"] = ps["sp_hist"][-Z_WIN:]

            td[key] = ps
            if len(ps["sp_hist"]) < MIN_HIST_FOR_TRADE:
                continue

            mu  = sum(ps["sp_hist"]) / len(ps["sp_hist"])
            var = sum((s - mu)**2 for s in ps["sp_hist"]) / len(ps["sp_hist"])
            sig = math.sqrt(var) if var > 1e-12 else 1e-9
            z   = (sp_now - mu) / sig

            y_pos = state.position.get(Y, 0)
            x_pos = state.position.get(X, 0)

            y_ords, x_ords = [], []

            # ── ENTRY: SHORT spread (z > +entry_z) ──
            #   spread above mean → sell Y, buy β·X (sign of β determines X side)
            if z > entry_z and y_pos > -POS_LIMIT:
                qty_y = _max_qty(base_qty, y_pos, x_pos, beta, "short")
                if qty_y > 0:
                    hedge_x = int(round(beta * qty_y))
                    y_bid = _best_bid(state, Y)
                    if y_bid is not None:
                        y_ords.append(Order(Y, int(y_bid), -qty_y))
                    if hedge_x > 0:
                        x_ask = _best_ask(state, X)
                        if x_ask is not None:
                            x_ords.append(Order(X, int(x_ask), hedge_x))
                    elif hedge_x < 0:
                        x_bid = _best_bid(state, X)
                        if x_bid is not None:
                            x_ords.append(Order(X, int(x_bid), hedge_x))

            # ── ENTRY: LONG spread (z < -entry_z) ──
            elif z < -entry_z and y_pos < POS_LIMIT:
                qty_y = _max_qty(base_qty, y_pos, x_pos, beta, "long")
                if qty_y > 0:
                    hedge_x = int(round(beta * qty_y))
                    y_ask = _best_ask(state, Y)
                    if y_ask is not None:
                        y_ords.append(Order(Y, int(y_ask), qty_y))
                    if hedge_x > 0:
                        x_bid = _best_bid(state, X)
                        if x_bid is not None:
                            x_ords.append(Order(X, int(x_bid), -hedge_x))
                    elif hedge_x < 0:
                        x_ask = _best_ask(state, X)
                        if x_ask is not None:
                            x_ords.append(Order(X, int(x_ask), -hedge_x))

            # ── EXIT: |z| crossed back near zero → flatten both legs ──
            elif abs(z) < exit_z:
                if y_pos > 0:
                    p = _best_bid(state, Y)
                    if p is not None: y_ords.append(Order(Y, int(p), -y_pos))
                elif y_pos < 0:
                    p = _best_ask(state, Y)
                    if p is not None: y_ords.append(Order(Y, int(p), -y_pos))
                if x_pos > 0:
                    p = _best_bid(state, X)
                    if p is not None: x_ords.append(Order(X, int(p), -x_pos))
                elif x_pos < 0:
                    p = _best_ask(state, X)
                    if p is not None: x_ords.append(Order(X, int(p), -x_pos))

            if y_ords: result.setdefault(Y, []).extend(y_ords)
            if x_ords: result.setdefault(X, []).extend(x_ords)

        return result, 0, json.dumps(td)
'''

# ── Step 3: Inject pairs and save ───────────────────────────────────────────
pairs_repr = '[\n' + ',\n'.join(f'    {p!r}' for p in SELECTED_PAIRS) + ',\n]'
trader_code = TRADER_SRC.replace('__PAIRS__', pairs_repr)

with open('round5_trader.py', 'w') as f:
    f.write(trader_code)

print(f'\n✅ Saved {os.path.abspath("round5_trader.py")}')
print(f'   Lines: {len(trader_code.splitlines())} | Pairs included: {len(SELECTED_PAIRS)}')
print('\nNext: run Cell 22b to validate in-sample PnL before uploading.')

In [ ]:
# %%
# ─── CELL 22b: In-Sample Trader Validator — Filter By Realised PnL ──────────
#
# Replicates the Trader's tick-by-tick rolling-β + z-score logic on the
# 3-day historical series. This is the LAST line of defence before upload:
# only deploy pairs that produce POSITIVE in-sample PnL with rolling β.
#
# ALPHA NOTE: The structural-break check (Cell 16b) flags β instability,
# but the in-sample simulator measures whether the Trader actually MAKES
# MONEY despite that instability. Some unstable pairs survive (mean
# reversion fast enough to outpace β drift); others bleed.

import math


def simulate_pair_trader(prices_df, Y, X, beta_init, entry_z, exit_z, qty,
                         beta_win=500, z_win=300, beta_refresh=50,
                         min_hist=150, max_beta_jump=2.0):
    """
    Tick-by-tick simulation matching the Trader's exact logic.
    Mark-to-market PnL on mid-price (no transaction cost).
    """
    y_arr = prices_df[Y].dropna().values
    x_arr = prices_df[X].dropna().values
    n = min(len(y_arr), len(x_arr))
    y_arr, x_arr = y_arr[:n], x_arr[:n]

    y_h, x_h, sp_h = [], [], []
    beta = beta_init
    y_pos = x_pos = 0
    cum_pnl = 0.0
    n_trades = 0
    pnl_track  = np.zeros(n)
    beta_track = np.full(n, np.nan)

    for t in range(n):
        y_h.append(y_arr[t]); x_h.append(x_arr[t])
        if len(y_h) > beta_win:
            y_h = y_h[-beta_win:]; x_h = x_h[-beta_win:]

        if (t + 1) % beta_refresh == 0 and len(y_h) >= 100:
            mx = sum(x_h)/len(x_h); my = sum(y_h)/len(y_h)
            sxx = sum((xi-mx)**2 for xi in x_h)
            syy = sum((yi-my)**2 for yi in y_h)
            sxy = sum((xi-mx)*(yi-my) for xi, yi in zip(x_h, y_h))
            disc = math.sqrt((sxx-syy)**2 + 4*sxy*sxy)
            lam_min = 0.5*((sxx+syy) - disc)
            denom = lam_min - sxx
            if abs(denom) > 1e-12:
                nb = -sxy / denom
                if abs(nb - beta) / max(abs(beta), 1e-9) < max_beta_jump:
                    beta = nb
        beta_track[t] = beta

        sp_now = y_arr[t] - beta * x_arr[t]
        sp_h.append(sp_now)
        if len(sp_h) > z_win:
            sp_h = sp_h[-z_win:]

        if len(sp_h) < min_hist:
            pnl_track[t] = cum_pnl
            continue

        mu  = sum(sp_h) / len(sp_h)
        var = sum((s-mu)**2 for s in sp_h) / len(sp_h)
        sig = math.sqrt(var) if var > 1e-12 else 1e-9
        z   = (sp_now - mu) / sig

        # Mark-to-market PnL from existing positions
        if t > 0:
            cum_pnl += y_pos * (y_arr[t] - y_arr[t-1]) + x_pos * (x_arr[t] - x_arr[t-1])
        pnl_track[t] = cum_pnl

        # ENTRY: short spread
        if z > entry_z and y_pos > -10:
            cap_y = 10 + y_pos
            if abs(beta) < 1e-9:
                cap_x = qty
            elif beta > 0:
                cap_x = (10 - x_pos) / beta
            else:
                cap_x = (10 + x_pos) / abs(beta)
            q = max(0, min(qty, int(cap_y), int(cap_x)))
            if q > 0:
                y_pos -= q
                x_pos += int(round(beta * q))
                n_trades += 1
        # ENTRY: long spread
        elif z < -entry_z and y_pos < 10:
            cap_y = 10 - y_pos
            if abs(beta) < 1e-9:
                cap_x = qty
            elif beta > 0:
                cap_x = (10 + x_pos) / beta
            else:
                cap_x = (10 - x_pos) / abs(beta)
            q = max(0, min(qty, int(cap_y), int(cap_x)))
            if q > 0:
                y_pos += q
                x_pos -= int(round(beta * q))
                n_trades += 1
        # EXIT
        elif abs(z) < exit_z and (y_pos != 0 or x_pos != 0):
            y_pos = 0; x_pos = 0; n_trades += 1

    return {
        'final_pnl':   cum_pnl,
        'n_trades':    n_trades,
        'pnl_track':   pnl_track,
        'beta_track':  beta_track,
        'beta_min':    float(np.nanmin(beta_track)),
        'beta_max':    float(np.nanmax(beta_track)),
        'final_y_pos': y_pos,
        'final_x_pos': x_pos,
    }


# ── Run simulation on every SELECTED_PAIRS member ──────────────────────────
print('=== In-sample Trader simulation (3-day mid-price MtM) ===')
print(f'Per-pair settings: BETA_WIN=500, Z_WIN=300, BETA_REFRESH=50, qty=5\n')

sim_results = []
for (Y, X, b0, ez, xz, q) in SELECTED_PAIRS:
    sim = simulate_pair_trader(prices, Y, X, b0, ez, xz, q)
    sim_results.append({
        'Y':         Y,
        'X':         X,
        'pair':      f'{Y.split("_")[-1]}/{X.split("_")[-1]}',
        'beta_init': round(b0, 4),
        'beta_min':  round(sim['beta_min'], 4),
        'beta_max':  round(sim['beta_max'], 4),
        'pnl':       round(sim['final_pnl'], 1),
        'trades':    sim['n_trades'],
        'sim':       sim,   # keep for plotting
    })

sim_df = pd.DataFrame(sim_results).sort_values('pnl', ascending=False)
print('=== Per-pair PnL (sorted) ===')
display(sim_df[['pair','beta_init','beta_min','beta_max','pnl','trades']])

# ── Plot β + cumulative PnL per pair ───────────────────────────────────────
n_p = len(sim_results)
if n_p > 0:
    fig, axes = plt.subplots(n_p, 2, figsize=(18, 3.5 * n_p))
    if n_p == 1: axes = axes.reshape(1, 2)
    for i, r in enumerate(sim_results):
        sim = r['sim']
        # rolling β
        axes[i, 0].plot(sim['beta_track'], lw=0.5, color='purple')
        axes[i, 0].axhline(r['beta_init'], color='red', ls='--', lw=1,
                            label=f"β_init={r['beta_init']:.3f}")
        axes[i, 0].axhline(0, color='black', lw=0.6)
        # day boundaries (10K ticks per day)
        for d in [10000, 20000]:
            axes[i, 0].axvline(d, color='gray', lw=0.6, ls=':')
        axes[i, 0].set_title(f"Rolling β: {r['pair']}", fontsize=9, fontweight='bold')
        axes[i, 0].legend(fontsize=7)
        # PnL
        c = 'darkgreen' if r['pnl'] > 0 else 'darkred'
        axes[i, 1].plot(sim['pnl_track'], lw=0.7, color=c)
        axes[i, 1].fill_between(range(len(sim['pnl_track'])), sim['pnl_track'],
                                 alpha=0.15, color=c)
        axes[i, 1].axhline(0, color='black', lw=0.6)
        for d in [10000, 20000]:
            axes[i, 1].axvline(d, color='gray', lw=0.6, ls=':')
        axes[i, 1].set_title(
            f"PnL: {r['pnl']:.0f}  Trades: {r['trades']}  "
            f"{'✓ DEPLOY' if r['pnl'] > 0 else '✗ DROP'}",
            fontsize=9, fontweight='bold', color=c)

    plt.suptitle('Trader Simulation — Per-Pair Rolling β + Cum PnL',
                 fontsize=12, y=1.001)
    plt.tight_layout()
    plt.savefig('trader_simulation.png', bbox_inches='tight')
    plt.show()

# ── Filter SELECTED_PAIRS to only positive-PnL pairs and regenerate Trader ──
profitable = sim_df[sim_df['pnl'] > 0]
DEPLOY_PAIRS = [
    next(p for p in SELECTED_PAIRS if p[0] == row['Y'] and p[1] == row['X'])
    for _, row in profitable.iterrows()
]

print(f'\n=== DEPLOYMENT FILTER ===')
print(f'Profitable in-sample: {len(DEPLOY_PAIRS)} / {len(SELECTED_PAIRS)} pairs')
for p in DEPLOY_PAIRS:
    print(f'  ✓ {p[0]} / {p[1]}  β_init={p[2]:.4f}')

# Regenerate round5_trader.py with ONLY profitable pairs
if len(DEPLOY_PAIRS) > 0:
    pairs_repr = '[\n' + ',\n'.join(f'    {p!r}' for p in DEPLOY_PAIRS) + ',\n]'
    trader_code_filtered = TRADER_SRC.replace('__PAIRS__', pairs_repr)
    with open('round5_trader.py', 'w') as f:
        f.write(trader_code_filtered)
    print(f'\n✅ Re-saved round5_trader.py with {len(DEPLOY_PAIRS)} profitable pairs.')
    print(f'   Total in-sample PnL: {sim_df[sim_df["pnl"] > 0]["pnl"].sum():.0f}')
else:
    print('\n⚠️  No profitable pairs — re-tune entry_z / qty before submission.')

In [ ]:
# %%
# ─── CELL 19: Special Analysis — UV Visors / Translators (Spectrum) ─────────
#
# Both categories follow a physical spectrum ordering.
# UV Visors: Yellow→Amber→Orange→Red→Magenta (wavelength ordering)
# Translators: Space Gray→Astro Black→Eclipse Charcoal→Graphite Mist→Void Blue
# Test whether adjacent spectrum members are more cointegrated than distant ones.

for cat in ['UV Visors', 'Translators']:
    cols = CATEGORIES[cat]
    print(f'\n=== {cat}: Pairwise ADF p-values (hedge-ratio spread) ===')
    
    mat = pd.DataFrame(index=[c.split('_')[-1] for c in cols],
                       columns=[c.split('_')[-1] for c in cols], dtype=float)
    
    for y_col, x_col in itertools.combinations(cols, 2):
        _, sp = fit_hedge_ratio(prices[y_col].dropna(), prices[x_col].dropna())
        p = adf_test(sp)['adf_pval']
        sy, sx = y_col.split('_')[-1], x_col.split('_')[-1]
        mat.loc[sy, sx] = p
        mat.loc[sx, sy] = p

    mat.values[np.eye(len(cols), dtype=bool)] = 0

    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    sns.heatmap(mat.astype(float), ax=ax, annot=True, fmt='.3f', cmap='RdYlGn_r',
                vmin=0, vmax=0.1, square=True, cbar_kws={'label': 'ADF p-value'})
    ax.set_title(f'{cat} — ADF p-value heatmap (green = stationary spread)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{cat.lower().replace(" ","_")}_adf_heatmap.png', bbox_inches='tight')
    plt.show()
    display(mat.astype(float).round(4))

In [ ]:
# %%
# ─── CELL 20: Final Signal Table & Strategy Summary ─────────────────────────
#
# Produce the final ranked list of tradeable pairs with all metrics,
# ready for strategy selection.

final_df = bt_df.merge(
    pair_df[['category','y','x','adf_pval','half_life','hurst','hedge_beta']],
    on=['category','y','x'], how='left'
).sort_values('total_pnl', ascending=False)

print('===================================================================')
print('         FINAL RANKED PAIRS — STRATEGY SELECTION TABLE')
print('===================================================================')
print('Criteria: high PnL + low ADF p-val + short half-life + H < 0.5')
print('Position limit per leg: ±10 | Tested qty per leg: ±5')
print('-------------------------------------------------------------------')
display(final_df[['category','y','x','total_pnl','sharpe','max_dd','n_trades',
                   'adf_pval','half_life','hurst','hedge_beta']].head(20).round(4))

print('\n=== Best pair per category (by total PnL) ===')
display(
    final_df.groupby('category')
             .apply(lambda g: g.nlargest(1, 'total_pnl').iloc[0])
             .reset_index(drop=True)
    [['category','y','x','total_pnl','sharpe','adf_pval','half_life','hurst']].round(4)
)

In [ ]:
# %%
# ─── CELL 25: Executive Summary — Refactored Pipeline + Alpha Findings ──────

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║       IMC PROSPERITY 4 — ROUND 5  (REFACTORED PIPELINE SUMMARY)         ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  PIPELINE CHANGES                                                        ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  • All simple OLS regressions: closed-form NumPy formulas; TLS via SVD.      ║
║    statsmodels.OLS removed; adfuller kept only for unit-root test.       ║
║  • TLS hedge ratio added via SVD (Cell 6)                                ║
║  • Rolling β: vectorized via sliding_window_view + closed-form OLS     ║
║    (Cell 16) → ~4x faster on 30K ticks, scales to all pairs in parallel ║
║  • New diagnostics: TLS comparison (7b), structural break (16b),         ║
║    bid-ask diagnostics (16c), friction-ratio filter (16d)                ║
║                                                                          ║
║  ★ ALPHA FINDING #1 — TLS DOMINATES OLS                                 ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  Empirically TLS produces a more stationary spread on ~80% of pairs.    ║
║  Examples (OLS p → TLS p):                                               ║
║    VANILLA / RASPBERRY:    0.037 → 0.001  (33x improvement)             ║
║    CHOCOLATE / RASPBERRY:  0.067 → 0.001  (47x improvement)             ║
║    STRAWBERRY / RASPBERRY: 0.165 → 0.010  (jumps to top-tier!)          ║
║  → For each tradable pair, USE THE TLS β, not the OLS β                 ║
║  → RASPBERRY emerges as a hub asset; β >> 1 ratios reveal it             ║
║    co-moves with all flavors at large amplitude.                         ║
║                                                                          ║
║  ★ ALPHA FINDING #2 — STRUCTURAL BREAKS ARE WIDESPREAD                  ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  Cointegration on the FULL 3-day series is misleading. Day-by-day β     ║
║  values are unstable for the top pairs:                                  ║
║    MICROCHIP_SQUARE / RECTANGLE:  β = [-1.13, -1.53, +0.07]  ⚠         ║
║    SNACKPACK_PISTACHIO / STRAWBERRY: β = [-0.04, +0.49, +0.51]  ⚠      ║
║  → Trading with a STATIC β computed on training data WILL FAIL on       ║
║    live data when the regime flips.                                      ║
║  → MITIGATION: use rolling β (W = 2000), re-estimate every 50-200 ticks ║
║  → DEPLOY: Cell 16's vectorized rolling_ols_vectorized() inside Trader  ║
║                                                                          ║
║  ★ ALPHA FINDING #3 — FRICTION IS NOT THE BOTTLENECK                    ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  All top pairs pass the 40% friction filter (typical ratios 0.04-0.22).  ║
║  Tightest spreads vs cost:                                               ║
║    SQUARE / RECTANGLE:   friction = 0.046  (cleanest)                   ║
║    OVAL / TRIANGLE:      friction = 0.040  (cleanest)                   ║
║    VACUUMING / LAUNDRY:  friction = 0.050                               ║
║  → Bid-ask is NOT the problem; β instability is the problem.            ║
║  → Slack budget for partial-fill / queue jumping exists.                 ║
║                                                                          ║
║  STRATEGY DEPLOYMENT TIERS                                               ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  TIER 1 (Static β, low risk):  ONLY pairs with structural_status=STABLE ║
║                                AND friction <= 0.30                      ║
║                                → tiny universe; verify with Cell 16d     ║
║  TIER 2 (Rolling β, main):     pairs with status=WATCH or STABLE        ║
║                                AND TLS p < 0.01                          ║
║                                → trade with rolling TLS β every tick     ║
║  TIER 3 (DO NOT TRADE):        status=BREAK regardless of ADF p-value   ║
║                                                                          ║
║  IMPLEMENTATION REMINDERS                                                ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  1. Use TLS β (not OLS β) for all production hedge ratios               ║
║  2. Recompute rolling β on every tick OR every K ticks (K<=200)         ║
║  3. ALWAYS int() wrap Order prices — float prices are REJECTED          ║
║  4. Maintain spread history in traderData (JSON list, capped at W)      ║
║  5. Z-score from rolling β-spread, not from static β-spread             ║
║  6. Position limit per leg: 10. Round-trip qty <= 5 per leg.            ║
║  7. Max 2 pairs per category to limit correlated risk                   ║
║  8. Stop trading a pair if rolling-β CV exceeds 0.30 in last 1000 ticks ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# %%
# ─── CELL 22: Trader Code Skeleton ──────────────────────────────────────────
#
# After selecting your best pairs from the analysis above,
# fill in SELECTED_PAIRS and run this cell to get a ready-to-submit
# Trader skeleton that implements the z-score pair trading strategy.
#
# Fill this manually based on cells 13-21:

SELECTED_PAIRS = [
    # (Y_product, X_product, hedge_beta, entry_z, exit_z, qty)
    # Example (replace with your best pairs from pair_df / final_df):
    # ('PEBBLES_XS', 'PEBBLES_XL', 1.00, 1.5, 0.3, 5),
    # ('UV_VISOR_YELLOW', 'UV_VISOR_MAGENTA', 0.98, 1.5, 0.3, 5),
]

code = '''
# ─── IMC Prosperity 4 — Round 5 Pair Trading Trader ───────────────────────
# Auto-generated skeleton. Fill SELECTED_PAIRS from analysis notebook.
# Position limit per product: 10.

from datamodel import OrderDepth, TradingState, Order
from typing import List
import json, math

# Each tuple: (Y, X, beta, entry_z, exit_z, qty)
SELECTED_PAIRS = [
'''
for p in SELECTED_PAIRS:
    code += f'    {p},\n'
code += '''
]

POS_LIMIT   = 10
ROLL_WIN    = 300   # lookback for rolling z-score


class Trader:
    def run(self, state: TradingState):
        result  = {}  # product -> List[Order]
        traderData = json.loads(state.traderData) if state.traderData else {}

        for (Y, X, beta, entry_z, exit_z, qty) in SELECTED_PAIRS:
            # ── Get mid prices ──────────────────────────────────────────────
            y_mid = self._mid(state, Y)
            x_mid = self._mid(state, X)
            if y_mid is None or x_mid is None:
                continue

            # ── Update spread history ────────────────────────────────────────
            key = f"{Y}_{X}_hist"
            hist = traderData.get(key, [])
            spread_now = y_mid - beta * x_mid
            hist.append(spread_now)
            if len(hist) > ROLL_WIN:
                hist = hist[-ROLL_WIN:]
            traderData[key] = hist

            if len(hist) < 30:  # need enough history
                continue

            # ── Compute z-score ──────────────────────────────────────────────
            mu  = sum(hist) / len(hist)
            var = sum((h - mu)**2 for h in hist) / len(hist)
            sig = math.sqrt(var) if var > 0 else 1e-9
            z   = (spread_now - mu) / sig

            # ── Current positions ────────────────────────────────────────────
            y_pos = state.position.get(Y, 0)
            x_pos = state.position.get(X, 0)

            # ── Signal & order generation ────────────────────────────────────
            y_orders, x_orders = [], []

            if z > entry_z and y_pos >= 0 and x_pos <= 0:  # SHORT spread
                # sell Y, buy X
                sell_y = min(qty, POS_LIMIT + y_pos)  # how much we can sell
                buy_x  = min(qty, POS_LIMIT - x_pos)  # but wait: we WANT negative x
                sell_y = min(qty, POS_LIMIT + y_pos)
                buy_x  = min(qty, POS_LIMIT + x_pos)  # x_pos <=0 so this increases |x_pos|
                # Actually: shorting spread = sell Y (y_pos goes negative) + buy X (x_pos goes positive)
                sell_y = min(qty, POS_LIMIT + y_pos)   # room to sell = limit + current pos (y_pos>=0)
                buy_x  = min(qty, POS_LIMIT - x_pos)   # room to buy  = limit - current pos (x_pos<=0, so this is limit+|x_pos|)
                to_trade = min(sell_y, buy_x)
                if to_trade > 0:
                    y_bid = self._best_bid(state, Y)
                    x_ask = self._best_ask(state, X)
                    if y_bid and x_ask:
                        y_orders.append(Order(Y, y_bid, -to_trade))
                        x_orders.append(Order(X, x_ask,  to_trade))

            elif z < -entry_z and y_pos <= 0 and x_pos >= 0:  # LONG spread
                # buy Y, sell X
                buy_y  = min(qty, POS_LIMIT + y_pos)   # room to buy
                sell_x = min(qty, POS_LIMIT - x_pos)   # room to sell
                to_trade = min(buy_y, sell_x)
                if to_trade > 0:
                    y_ask = self._best_ask(state, Y)
                    x_bid = self._best_bid(state, X)
                    if y_ask and x_bid:
                        y_orders.append(Order(Y, y_ask,  to_trade))
                        x_orders.append(Order(X, x_bid, -to_trade))

            elif abs(z) < exit_z:  # EXIT / flatten
                if y_pos > 0:
                    y_bid = self._best_bid(state, Y)
                    if y_bid: y_orders.append(Order(Y, y_bid, -y_pos))
                elif y_pos < 0:
                    y_ask = self._best_ask(state, Y)
                    if y_ask: y_orders.append(Order(Y, y_ask, -y_pos))
                if x_pos > 0:
                    x_bid = self._best_bid(state, X)
                    if x_bid: x_orders.append(Order(X, x_bid, -x_pos))
                elif x_pos < 0:
                    x_ask = self._best_ask(state, X)
                    if x_ask: x_orders.append(Order(X, x_ask, -x_pos))

            if y_orders: result[Y] = result.get(Y, []) + y_orders
            if x_orders: result[X] = result.get(X, []) + x_orders

        return result, 0, json.dumps(traderData)

    @staticmethod
    def _mid(state: TradingState, product: str):
        od = state.order_depths.get(product)
        if od is None: return None
        if not od.buy_orders or not od.sell_orders: return None
        return (max(od.buy_orders) + min(od.sell_orders)) / 2

    @staticmethod
    def _best_bid(state: TradingState, product: str):
        od = state.order_depths.get(product)
        if od is None or not od.buy_orders: return None
        return int(max(od.buy_orders))

    @staticmethod
    def _best_ask(state: TradingState, product: str):
        od = state.order_depths.get(product)
        if od is None or not od.sell_orders: return None
        return int(min(od.sell_orders))
'''

print(code)

# Optionally save to file
with open('round5_trader_skeleton.py', 'w') as f:
    f.write(code)
print('\nSaved: round5_trader_skeleton.py')

In [ ]:
# %%
# ─── CELL 23: Category-Level Mean Reversion Check (Basket vs Single) ────────
#
# Alternative to pure pairs: trade ONE asset vs the EQUAL-WEIGHTED basket
# of the remaining 4. This is more robust when no single pair dominates.

basket_results = []

for cat, cols in CATEGORIES.items():
    cat_df = prices[cols].dropna()

    for target in cols:
        others = [c for c in cols if c != target]
        basket = cat_df[others].mean(axis=1)  # equal-weighted basket

        _, spread = fit_hedge_ratio(cat_df[target], basket)
        adf = adf_test(spread)
        hl  = half_life(spread)
        hu  = hurst_exponent(spread)

        basket_results.append({
            'category':  cat,
            'target':    target,
            'basket':    '+'.join([c.split('_')[-1] for c in others]),
            'adf_pval':  round(adf['adf_pval'], 5),
            'half_life': round(hl, 1) if np.isfinite(hl) else 9999,
            'hurst':     round(hu, 4),
        })

basket_df = pd.DataFrame(basket_results).sort_values('adf_pval')
print('=== 1 Asset vs. 4-Asset Equal-Weighted Basket (within category) ===')
print('Stationary at 5%:', (basket_df['adf_pval'] < 0.05).sum())
display(basket_df.head(20))

In [ ]:
# %%
# ─── CELL 24: Cross-Day Cointegration Stability ──────────────────────────────
#
# Run EG tests on each individual day to check if the relationship
# is stable across time — fragile pairs (only cointegrated on 1 day)
# should be deprioritized.

print('=== Cointegration stability across days ===')
print('(A pair should be stationary on all 3 days to be trusted)\n')

# Focus on top 15 pairs from global analysis
top15 = pair_df.head(15)
stability_rows = []

for _, row in top15.iterrows():
    rec = {'category': row['category'], 'y': row['y'], 'x': row['x'],
           'global_adf': row['adf_pval']}
    for day in DAYS:
        y_d = day_slices[day][row['y']].dropna()
        x_d = day_slices[day][row['x']].dropna()
        common = y_d.index.intersection(x_d.index)
        if len(common) < 200:
            rec[f'day{day}_pval'] = np.nan
            continue
        _, sp = fit_hedge_ratio(y_d.loc[common], x_d.loc[common])
        rec[f'day{day}_pval'] = round(adf_test(sp)['adf_pval'], 4)
    stability_rows.append(rec)

stab_df = pd.DataFrame(stability_rows)
stab_df['n_days_stationary'] = (
    (stab_df[f'day{DAYS[0]}_pval'] < 0.05).astype(int) +
    (stab_df[f'day{DAYS[1]}_pval'] < 0.05).astype(int) +
    (stab_df[f'day{DAYS[2]}_pval'] < 0.05).astype(int)
)
display(stab_df.sort_values('n_days_stationary', ascending=False))

In [ ]:
# %%
# ─── CELL 25: Executive Summary — Empirical Findings ────────────────────────

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║         IMC PROSPERITY 4 — ROUND 5 EMPIRICAL FINDINGS SUMMARY           ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  TOP CONFIRMED COINTEGRATED PAIRS (ADF p < 0.05, all 3 days)            ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  RANK  PAIR                              ADF-p    HL(ticks)  NOTE        ║
║   #1   MICROCHIP_SQUARE / RECTANGLE      0.007      819      ★ BEST     ║
║   #2   SNACKPACK_PISTACHIO / STRAWBERRY  0.008      806      ★          ║
║   #3   SNACKPACK_CHOCOLATE / STRAWBERRY  0.009      802      ★          ║
║   #4   SNACKPACK_CHOCOLATE / PISTACHIO   0.012      876                  ║
║   #5   SNACKPACK_VANILLA / PISTACHIO     0.015      879                  ║
║   #6   MICROCHIP_OVAL / TRIANGLE         0.017     1059                  ║
║   #7   ROBOT_VACUUMING / LAUNDRY         0.020     1082                  ║
║   #8   SLEEP_POD_LAMB_WOOL / NYLON       0.022     1198                  ║
║   #9   UV_VISOR_AMBER / MAGENTA          0.023     1058                  ║
║  #10   SNACKPACK_VANILLA / STRAWBERRY    0.023      969                  ║
║  #11   OXYGEN_SHAKE_CHOCOLATE / GARLIC   0.029     1418                  ║
║  #12   ROBOT_VACUUMING / DISHES          0.030      853                  ║
║  #13   SLEEP_POD_POLYESTER / COTTON      0.031     1138                  ║
║  #14   GALAXY_SOUNDS_DARK_MATTER / RINGS 0.037     1149                  ║
║  #15   TRANSLATOR_CHARCOAL / VOID_BLUE   0.041     1240                  ║
║                                                                          ║
║  KEY SURPRISES vs INITIAL HYPOTHESES                                     ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  ✗ Panels 2X2 vs 1X4 (same area=4): NOT cointegrated (p=0.376)          ║
║    → Area hypothesis was wrong; panels not priced by area                ║
║  ✗ Pebbles size gradient: Only borderline (XS/S p=0.07)                 ║
║    → Weaker than expected; avoid unless cross-day stable                 ║
║  ✓ Snacks: STRONGEST category — 4+ stationary pairs at <2.5%            ║
║    → PISTACHIO/STRAWBERRY/CHOCOLATE/VANILLA cluster cointegrated         ║
║  ✓ Microchips SQUARE/RECTANGLE: #1 overall                              ║
║  ✓ UV Visors AMBER/MAGENTA (skip-2): stronger than adjacent pairs        ║
║                                                                          ║
║  CATEGORY TRADABILITY RANKING                                            ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  Snacks      ★★★  Multiple pairs; pick 2 (PISTACHIO/STRAWBERRY + 1)     ║
║  Microchips  ★★★  SQUARE/RECTANGLE (#1) + OVAL/TRIANGLE (#6)            ║
║  Robots      ★★   VACUUMING/LAUNDRY + VACUUMING/DISHES                  ║
║  Sleep Pods  ★★   WOOL/NYLON + POLYESTER/COTTON                         ║
║  UV Visors   ★★   AMBER/MAGENTA only                                    ║
║  Translators ★    CHARCOAL/VOID_BLUE borderline (p=0.041)               ║
║  Galaxy      ★    DARK_MATTER/RINGS borderline (p=0.037)                ║
║  Shakes      ★    CHOCOLATE/GARLIC only (p=0.029)                       ║
║  Pebbles     -    Weak; skip or deprioritize                            ║
║  Panels      -    No cointegration found; skip                          ║
║                                                                          ║
║  TRADING RULES                                                           ║
║  ──────────────────────────────────────────────────────────────────────  ║
║  1. Only trade pairs with adf_pval < 0.05 AND n_days_stat >= 2          ║
║  2. Half-life target: 500-2000 ticks (too fast = slippage)              ║
║  3. Entry z = 1.5-2.0, exit z = 0.0-0.3 (tune per pair in Cell 15)     ║
║  4. qty per leg <= 5 (leaves room for position building to 10)          ║
║  5. Use rolling beta (window=2000) for regime changes (Cell 16)         ║
║  6. ALWAYS int() wrap Order prices — float prices are REJECTED          ║
║  7. Store spread history in traderData as JSON list                     ║
║  8. For Snacks cluster: limit to 2 pairs max (correlated risk)          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")